# Crop Actor Holdout Experiments

Ce notebook reprend le script `crop_actor_holdout_experiments.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Robustesse acteur des modeles crop utilises dans le contexte live.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Actor-held-out audit for attention and blouse/PPE crop classifiers.
- Commande de reproduction referencee : crop actor holdout.
- Artefacts controles : Actor-held-out attention/blouse crop robustness audit exists. (`runs/exp_066_crop_actor_holdout/metrics/crop_actor_holdout_summary.csv`).
- Run par defaut : `runs/exp_066_crop_actor_holdout`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "crop_actor_holdout_experiments.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

from crop_cnn_experiments import CropDataset, TARGETS, predict, train_crop_model
from ml_pipeline import ROOT, safe_auc, write_json
from sequence_experiments import append_report, make_run_dir


## Fonction `resolve`

Cette cellule definit `resolve`. Elle prepare une partie du script.

In [ ]:
def resolve(path):
    path = Path(path)
    return path if path.is_absolute() else ROOT / path


## Fonction `attach_actor`

Cette cellule definit `attach_actor`. Elle prepare une partie du script.

In [ ]:
def attach_actor(index):
    videos = pd.read_csv(ROOT / "annotations" / "videos.csv")[["video_id", "actor"]]
    out = index.merge(videos, on="video_id", how="left")
    if out["actor"].isna().any():
        missing = sorted(out[out["actor"].isna()]["video_id"].unique().tolist())
        raise SystemExit(f"Missing actor labels for crop videos: {missing[:10]}")
    return out


## Fonction `split_remaining_actor_videos`

Cette cellule definit `split_remaining_actor_videos`. Elle prepare une partie du script.

In [ ]:
def split_remaining_actor_videos(video_df, held_actor, seed, val_fraction):
    remaining = video_df[video_df["actor"].ne(held_actor)].copy()
    if remaining.empty:
        raise SystemExit(f"No remaining actor videos for held_actor={held_actor}")
    labels = remaining["attention_label"].astype(str) + "_" + remaining["blouse_label"].astype(str)
    stratify = labels if labels.value_counts().min() >= 2 else None
    train_ids, val_ids = train_test_split(
        remaining["video_id"].to_numpy(),
        test_size=val_fraction,
        random_state=seed,
        stratify=stratify,
    )
    split = {video_id: "test" for video_id in video_df[video_df["actor"].eq(held_actor)]["video_id"].tolist()}
    split.update({video_id: "train" for video_id in train_ids})
    split.update({video_id: "val" for video_id in val_ids})
    return split


## Fonction `threshold_metrics`

Cette cellule definit `threshold_metrics`. Elle prepare une partie du script.

In [ ]:
def threshold_metrics(y, p):
    best = None
    for threshold in np.arange(0.05, 1.0, 0.05):
        pred = (p >= threshold).astype(int)
        item = {
            "threshold": float(threshold),
            "f1": float(f1_score(y, pred, zero_division=0)),
            "accuracy": float(accuracy_score(y, pred)),
            "balanced_accuracy": float(balanced_accuracy_score(y, pred)) if len(np.unique(y)) > 1 else np.nan,
            "confusion_matrix": confusion_matrix(y, pred, labels=[0, 1]).tolist(),
        }
        if best is None or item["f1"] > best["f1"]:
            best = item
    return best


## Fonction `evaluate_predictions`

Cette cellule definit `evaluate_predictions`. Elle prepare une partie du script.

In [ ]:
def evaluate_predictions(pred, target):
    rows = []
    for level in ["crop", "video"]:
        if level == "video":
            eval_df = (
                pred.groupby(["split", "video_id"], as_index=False)
                .agg(label=(f"{target}_label", "max"), risk=("risk", "mean"), actor=("actor", "first"))
                .copy()
            )
        else:
            eval_df = pred.rename(columns={f"{target}_label": "label"}).copy()
        for split, group in eval_df.groupby("split"):
            y = group["label"].astype(int).to_numpy()
            p = group["risk"].astype(float).to_numpy()
            best = threshold_metrics(y, p)
            rows.append(
                {
                    "level": level,
                    "split": split,
                    "n": int(len(group)),
                    "positive": int(y.sum()),
                    "average_precision": safe_auc(average_precision_score, y, p),
                    "roc_auc": safe_auc(roc_auc_score, y, p),
                    **best,
                }
            )
    return rows


## Fonction `summarize`

Cette cellule definit `summarize`. Elle prepare une partie du script.

In [ ]:
def summarize(metrics):
    metric_cols = ["average_precision", "roc_auc", "f1", "balanced_accuracy"]
    group_cols = ["held_actor", "target", "architecture", "level", "split"]
    rows = []
    for keys, group in metrics.groupby(group_cols):
        row = dict(zip(group_cols, keys))
        row["n_repeats"] = int(group["repeat_seed"].nunique())
        row["n_mean"] = float(group["n"].mean())
        row["positive_mean"] = float(group["positive"].mean())
        for col in metric_cols:
            row[f"{col}_mean"] = float(group[col].mean())
            row[f"{col}_std"] = float(group[col].std(ddof=0))
            row[f"{col}_min"] = float(group[col].min())
            row[f"{col}_max"] = float(group[col].max())
        rows.append(row)
    return pd.DataFrame(rows)


## Fonction `fmt`

Cette cellule definit `fmt`. Elle prepare une partie du script.

In [ ]:
def fmt(value, digits=3):
    if value is None or pd.isna(value):
        return "NA"
    return f"{float(value):.{digits}f}"


## Fonction `write_summary`

Cette cellule definit `write_summary`. Elle prepare une partie du script.

In [ ]:
def write_summary(run_dir, summary, split_counts):
    lines = ["# Crop Actor-Held-Out Audit", ""]
    lines.append("This audit tests whether attention and blouse/PPE crop classifiers generalize across actors. One actor is held out entirely as test; the remaining actor's parent videos are split into train/validation.")
    lines.append("")
    lines.append("## Split Counts")
    lines.append("")
    lines.append("| held actor | seed | split | videos | crops | attention positives | blouse positives |")
    lines.append("|---|---:|---|---:|---:|---:|---:|")
    for _, row in split_counts.sort_values(["held_actor", "repeat_seed", "split"]).iterrows():
        lines.append(
            f"| {row['held_actor']} | {int(row['repeat_seed'])} | {row['split']} | "
            f"{int(row['videos'])} | {int(row['crops'])} | {int(row['attention_positive_crops'])} | {int(row['blouse_positive_crops'])} |"
        )
    lines.append("")
    lines.append("## Held-Out Test Summary")
    lines.append("")
    lines.append("| held actor | target | level | rank | architecture | repeats | AP mean | AP std | ROC AUC | F1 | balanced acc |")
    lines.append("|---|---|---|---:|---|---:|---:|---:|---:|---:|---:|")
    test = summary[summary["split"].eq("test")].copy()
    for (held_actor, target, level), group in test.sort_values(
        ["held_actor", "target", "level", "average_precision_mean"],
        ascending=[True, True, True, False],
    ).groupby(["held_actor", "target", "level"]):
        for rank, (_, row) in enumerate(group.iterrows(), start=1):
            lines.append(
                f"| {held_actor} | {target} | {level} | {rank} | {row['architecture']} | {int(row['n_repeats'])} | "
                f"{fmt(row['average_precision_mean'])} | {fmt(row['average_precision_std'])} | "
                f"{fmt(row['roc_auc_mean'])} | {fmt(row['f1_mean'])} | {fmt(row['balanced_accuracy_mean'])} |"
            )
    lines.append("")
    lines.append("## Interpretation")
    lines.append("")
    lines.append("- These results should be treated as stress tests, not production actor robustness, because only two actors exist.")
    lines.append("- If one held-out actor collapses for attention or blouse/PPE, the final fused score inherits that actor-generalization limitation.")
    (run_dir / "crop_actor_holdout_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    crop_run = resolve(args.crop_run)
    source_index = pd.read_csv(crop_run / "features" / "crop_cnn_index.csv")
    source_index["path"] = source_index["path"].apply(lambda p: str(crop_run / p))
    source_index = attach_actor(source_index)
    video_df = (
        source_index.groupby("video_id", as_index=False)
        .agg(
            actor=("actor", "first"),
            attention_label=("attention_label", "max"),
            blouse_label=("blouse_label", "max"),
        )
        .copy()
    )
    actors = sorted(video_df["actor"].dropna().unique().tolist())
    if len(actors) < 2:
        raise SystemExit("Actor-held-out crop audit requires at least two actors.")

    run_dir = make_run_dir(args.run_name)
    write_json(
        run_dir / "config.json",
        {
            "crop_run": str(crop_run),
            "actors": actors,
            "held_actors": args.held_actors or actors,
            "seeds": args.seeds,
            "architectures": args.architectures,
            "epochs": args.epochs,
            "patience": args.patience,
            "val_fraction_of_remaining_actor_videos": args.val_fraction,
            "split_policy": "test is the held-out actor; train/val are parent-video splits from the remaining actor",
        },
    )

    device = torch.device("cuda" if torch.cuda.is_available() and args.device == "auto" else args.device)
    all_metrics = []
    all_history = []
    split_rows = []
    held_actors = args.held_actors or actors
    for held_actor in held_actors:
        for repeat_seed in args.seeds:
            split = split_remaining_actor_videos(video_df, held_actor, repeat_seed, args.val_fraction)
            index = source_index.copy()
            index["split"] = index["video_id"].map(split)
            split_path = run_dir / "features" / f"crop_actor_split_seed{repeat_seed}_holdout_{held_actor}.csv"
            index.to_csv(split_path, index=False)
            for split_name, group in index.groupby("split"):
                split_rows.append(
                    {
                        "held_actor": held_actor,
                        "repeat_seed": repeat_seed,
                        "split": split_name,
                        "videos": int(group["video_id"].nunique()),
                        "crops": int(len(group)),
                        "attention_positive_crops": int(group["attention_label"].sum()),
                        "blouse_positive_crops": int(group["blouse_label"].sum()),
                    }
                )
            for target in TARGETS:
                for arch in args.architectures:
                    print(f"training holdout={held_actor} seed={repeat_seed} {target} {arch} on {device}")
                    model, history, train_time_s, model_size = train_crop_model(run_dir, index, target, arch, args, device)
                    default_model_path = run_dir / "models" / f"{target}_{arch}.pt"
                    renamed = run_dir / "models" / f"{target}_seed{repeat_seed}_holdout_{held_actor}_{arch}.pt"
                    if default_model_path.exists():
                        default_model_path.replace(renamed)
                    for row in history:
                        row["repeat_seed"] = repeat_seed
                        row["held_actor"] = held_actor
                    all_history.extend(history)

                    loader = DataLoader(
                        CropDataset(run_dir, index, target, train=False, image_size=args.image_size),
                        batch_size=args.batch_size,
                        shuffle=False,
                        num_workers=0,
                    )
                    probs, _ = predict(model, loader, device)
                    pred = index[["video_id", "actor", "split", "frame", "time_s", f"{target}_label"]].copy()
                    pred["target"] = target
                    pred["architecture"] = arch
                    pred["repeat_seed"] = repeat_seed
                    pred["held_actor"] = held_actor
                    pred["risk"] = probs
                    pred.to_csv(run_dir / "features" / f"predictions_{target}_seed{repeat_seed}_holdout_{held_actor}_{arch}.csv", index=False)
                    for metric in evaluate_predictions(pred, target):
                        metric.update(
                            {
                                "held_actor": held_actor,
                                "repeat_seed": repeat_seed,
                                "target": target,
                                "architecture": arch,
                                "train_time_s": train_time_s,
                                "model_size_bytes": model_size,
                            }
                        )
                        all_metrics.append(metric)
                    pd.DataFrame(all_metrics).to_csv(run_dir / "metrics" / "crop_actor_holdout_metrics.csv", index=False)
                    pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "crop_actor_holdout_training_history.csv", index=False)

    metrics = pd.DataFrame(all_metrics)
    split_counts = pd.DataFrame(split_rows)
    summary = summarize(metrics)
    metrics.to_csv(run_dir / "metrics" / "crop_actor_holdout_metrics.csv", index=False)
    summary.to_csv(run_dir / "metrics" / "crop_actor_holdout_summary.csv", index=False)
    split_counts.to_csv(run_dir / "metrics" / "crop_actor_holdout_split_counts.csv", index=False)
    pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "crop_actor_holdout_training_history.csv", index=False)
    write_summary(run_dir, summary, split_counts)
    append_report(run_dir, "Crop Actor-Held-Out Audit", f"- Summary: `{run_dir / 'crop_actor_holdout_summary.md'}`")
    print(run_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Actor-held-out audit for attention and blouse/PPE crop classifiers.")
    parser.add_argument("--crop-run", default="runs/exp_013_crop_cnn_catalogue")
    parser.add_argument("--run-name", default="exp_066_crop_actor_holdout")
    parser.add_argument("--architectures", nargs="+", default=["small_cnn", "mobilenet_v3_small", "resnet18"])
    parser.add_argument("--seeds", nargs="+", type=int, default=[101, 202, 303])
    parser.add_argument("--held-actors", nargs="*", default=None)
    parser.add_argument("--val-fraction", type=float, default=0.25)
    parser.add_argument("--image-size", type=int, default=224)
    parser.add_argument("--epochs", type=int, default=8)
    parser.add_argument("--patience", type=int, default=2)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--lr", type=float, default=3e-4)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--no-pretrained", action="store_true")
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_066_crop_actor_holdout_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["crop_actor_holdout_experiments.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
